In [3]:
import os
from dotenv import load_dotenv
os.chdir("..")
from chatPrabhu.utils import logger

In [4]:
logger.info("Setting up RAG pipeline...")

In [8]:
#import embedding and vectorstore
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain.vectorstores import FAISS

In [13]:
embeddings=GoogleGenerativeAIEmbeddings(model="models/embedding-001")
vectorstore=FAISS.load_local("vectorstore",embeddings,allow_dangerous_deserialization=True)

In [18]:
from langchain.chat_models import init_chat_model
from langchain.chains import RetrievalQA

In [33]:
retriever = vectorstore.as_retriever(search_kwargs={"k": 1})

llm = init_chat_model("gemini-2.0-flash", model_provider="google_genai", temperature=0.3)

rag_chain = RetrievalQA.from_chain_type(
    llm=llm,
    retriever=retriever,
    return_source_documents=True
)

In [34]:
from langgraph.graph import StateGraph, END

# Define the graph state as a simple dictionary
class GraphState(dict):
    pass

# Define a node that runs the RAG chain
def rag_node(state):
    question = state.get("question")
    if not question:
        raise ValueError("No 'question' key found in state.")
    
    result = rag_chain({"query": question})
    return {
        "question": question,
        "answer": result["result"],
        "sources": result["source_documents"]
    }


graph = StateGraph(dict)
graph.add_node("RAG", rag_node)
graph.set_entry_point("RAG")
graph.add_edge("RAG", END)

compiled_graph = graph.compile()


In [35]:
query = "What are the key takeaways from the first document?"
response = compiled_graph.invoke({"question": "what are regulative principles"})

print("Answer:", response["answer"])
print("\nSources:")
for doc in response["sources"]:
    print(" -", doc.metadata["source"])

Answer: The four regulative principles are: no meat-eating, no intoxication, no gambling, and no illicit sex.

Sources:
 - data\philosophy.txt
